In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os

In [26]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dataclasses import dataclass
from typing_extensions import TypedDict
from typing import Dict,List,Any
from langchain.tools import tool,ToolRuntime
from langchain.agents.middleware import ToolCallRequest

In [ ]:
model = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from langchain_tavily import TavilyResearch

In [ ]:
search = TavilyResearch()

In [ ]:
from langchain.agents.middleware import ModelRequest,ModelResponse,wrap_model_call
@wrap_model_call
def logging_middleware(request:ModelRequest,handler):
    print(request.model)
    messages = request.state["messages"]
    print("messages",messages)
    
    handler_response = handler(request)
    
    print("handler_response",handler_response)
    return handler_response
    

In [ ]:
@wrap_model_call
def dynamic_model_middleware(request:ModelRequest,handler):
    complex_keywords = ["complex","multiple","detailed","analysis","explain"]
    
    #last msg
    messages = request.state["messages"] 
    last_message = messages[-1].content
    
    if any(keyword in last_message.lower() for keyword in complex_keywords):
        request.override(model= ChatOpenAI(model="gpt-5.1-mini"))
        print("switched to GPT 5.1 min")
    else:
        request.override(model= ChatOpenAI(model="gpt-4.5-mini"))    
        print("switch GPT 4.5")
        
    handler_response = handler(request)
    return handler_response   

In [30]:
@tool
def fetch_live_extra_data(topic:str)->str:
    """ 
            Fetch live or real time data using the tool for specific topic(weather,stock quote)
            use this when user asks for uptodate minute or live information
    """
    
    raise ConnectionError("simulated failure: External data serivice unreachble")

from langchain.messages import ToolMessage    
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def handle_tool_error(request:ToolCallRequest,handler):
    try:
        print(
            f"Executing tool {request.tool.name} "
            f"with input: {request.tool_call}"
        )
        return handler(request)
    except Exception as e:
        
        return ToolMessage(
            content=f"Error while executing {request.tool.name}: {e}",
            tool_call_id=request.tool_call["id"]
        )
            
        

In [31]:
agent = create_agent(
    
    model=model,
    tools=[fetch_live_extra_data],
    middleware=[handle_tool_error],
    system_prompt=""" 
    1 you are a helpful chat assistant
    2.fetch_live_extra_data use this when user asks live or real time data or external data
    3.use tavily search tool for general web search, so if tool search is service error offer
    
    
    """
)

In [32]:
response = agent.invoke({
    "messages":[
        {"role":"user","content":"what is temparature in seattle right now"}
    ]
})

Executing tool fetch_live_extra_data with input: {'name': 'fetch_live_extra_data', 'args': {'topic': 'temperature in Seattle'}, 'id': 'call_OJSHRko2Gibg35wxvGYhemkT', 'type': 'tool_call'}


In [34]:
response

{'messages': [HumanMessage(content='what is temparature in seattle right now', additional_kwargs={}, response_metadata={}, id='d4ad3fc0-9aee-4e3a-8c9c-debcc423751c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 135, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_372aa90621', 'id': 'chatcmpl-EQZay0d4zaTZdTZPXsoG56M3s1KPe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c46f-eadd-77a0-985d-595582a0b57e-0', tool_calls=[{'name': 'fetch_live_extra_data', 'args': {'topic'

In [38]:
response["messages"][-2].tool_calls

AttributeError: 'ToolMessage' object has no attribute 'tool_calls'